# evaluation/02 — SDM Comparison

Evaluates all five ANTHEIA models using SDM-derived pollinator activity
curves instead of GBIF-derived ones.

## What SDM Replaces

In the primary results (`evaluation/01_seed_testing.ipynb`), pollinator
activity curves (`a_curves`) are built from GBIF occurrence records —
normalized weekly observation histograms. These are biased by sampling
effort: peaks reflect when iNaturalist users were active, not when
pollinators were active.

SDM (Species Distribution Model) activity curves are model-predicted,
produced by Dan Cher's species distribution models independently of
observation effort. They provide the pollinator-side analog of what PPE
provides on the plant side.

## Current Status

Dan's SDM currently covers **1,615 pollinator species** — a small test
set. After intersection with Vp, this yields **1,275 pollinators** and
**122 positive pairs** (vs. 139 under GBIF).

The smaller pair universe accounts for the lower absolute performance
of SDM models relative to GBIF models.

**This notebook will need to be rerun when the full 25,466-species SDM
becomes available. That is the first task for September.**

## Key Technical Note

SDM curves use a coordinate offset: `lat_bin = centroid_lat - 0.25`,
`lon_bin = centroid_lon - 0.25`. This maps SDM grid centroids to the
same bin corner convention as the F/P matrices.

## Expected Results (preliminary, 1,615-species SDM)

| Model | ROC-AUC | PR-AUC |
|---|---|---|
| Spatial Baseline-SDM | 0.9212 ± 0.0332 | 0.8628 ± 0.0513 |
| ANTHEIA-Scalar-SDM | 0.9235 ± 0.0320 | 0.8658 ± 0.0489 |
| ANTHEIA-4D-SDM | 0.9196 ± 0.0340 | 0.8716 ± 0.0470 |
| ANTHEIA-15D-SDM | 0.9334 ± 0.0319 | 0.8901 ± 0.0542 |
| **ANTHEIA-PMf-SDM** | **0.9489 ± 0.0245** | **0.8911 ± 0.0480** |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from pathlib import Path

BASE     = Path("/scratch/ariana.l")
OLD_S4   = BASE / "Stage 4 Link Prediction Model"
NEW_S4   = BASE / "New Stage 4 Link Prediction Model"
STAGE5   = BASE / "Stage 5 PPE Representation Study"
STAGE6   = BASE / "Stage 6 Seed Testing"

VF_PATH       = OLD_S4 / "stage4_Vf_phenofield.csv"
F_PATH        = OLD_S4 / "stage4_F_existence_phenofield.csv"
VP_PATH       = NEW_S4 / "stage4_Vp_corrected.csv"
P_PATH        = NEW_S4 / "stage4_P_existence_corrected.csv"
GLOBI_PATH    = OLD_S4 / "stage4_globi_conus_broad.csv"
F_CURVES_PATH = NEW_S4 / "f_curves_ppe.csv"
VDELTA_4D     = STAGE5 / "stage5_Vdelta_ppe.csv"
VDELTA_15D    = STAGE5 / "stage5_Vdelta_15d.csv"
VF_PROB       = STAGE5 / "stage5_Vf_prob.csv"

# SDM inputs
SDM_PATH      = STAGE6 / "pollinator_activity_curves.parquet"
MANIFEST_PATH = STAGE6 / "species_manifest.csv"

SEEDS     = [42, 0, 1, 2, 3]
NEG_RATIO = 3

print("Paths OK")

In [ ]:
# Load all plant-side embeddings and curves (same as primary results)
print("Loading plant side...")
Vf_df   = pd.read_csv(VF_PATH,   index_col=0)
F_df    = pd.read_csv(F_PATH,    index_col=0)
Vd4_df  = pd.read_csv(VDELTA_4D,  index_col=0)
Vd15_df = pd.read_csv(VDELTA_15D, index_col=0)
Vfp_df  = pd.read_csv(VF_PROB,    index_col=0)
f_curves_df = pd.read_csv(F_CURVES_PATH, index_col=0)
f_curves_df.columns = list(range(52))

print("Loading pollinator side...")
Vp_df = pd.read_csv(VP_PATH, index_col=0)
P_df  = pd.read_csv(P_PATH,  index_col=0)

common_bins = [b for b in F_df.columns if b in set(P_df.columns)]
F_common = F_df[common_bins].values
P_common = P_df[common_bins].values
fc_idx   = {sp: i for i, sp in enumerate(F_df.index)}
pc_idx   = {sp: i for i, sp in enumerate(P_df.index)}

def compute_N(plant, pollinator):
    return float(F_common[fc_idx[plant]] @ P_common[pc_idx[pollinator]])

print(f"Common bins: {len(common_bins)}")

In [ ]:
# Build SDM activity curves
# SDM parquet contains per-species, per-bin, per-week activity predictions
# Key difference from GBIF: model-predicted, not observation-derived

print("Loading SDM parquet and manifest...")
manifest = pd.read_csv(MANIFEST_PATH)
id_to_species = dict(zip(manifest["species_id"], manifest["species"]))
print(f"  Manifest species: {len(manifest)}")

sdm = pd.read_parquet(
    SDM_PATH,
    columns=["species_id", "centroid_lat", "centroid_lon", "week", "activity_norm"]
)
print(f"  SDM shape: {sdm.shape}")

# Coordinate offset: centroid → bin corner
# SDM uses grid centroids; F/P matrices use bin corners
# Correction: lat_bin = centroid_lat - 0.25, lon_bin = centroid_lon - 0.25
common_bins_set = set(common_bins)

sdm["bin"] = [
    f"{round(la - 0.25, 1)}_{round(lo - 0.25, 1)}"
    for la, lo in zip(sdm["centroid_lat"], sdm["centroid_lon"])
]
sdm = sdm[sdm["bin"].isin(common_bins_set)]
sdm["week"]    = sdm["week"].clip(0, 51).astype(int)
sdm["species"] = sdm["species_id"].map(id_to_species)
sdm = sdm.dropna(subset=["species"])
print(f"  SDM after CONUS filter: {sdm.shape}")

# Aggregate to per-species 52-week curve
a_sdm = (
    sdm.groupby(["species", "week"])["activity_norm"]
    .mean()
    .unstack(fill_value=0)
    .reindex(columns=list(range(52)), fill_value=0)
)
row_sums = a_sdm.sum(axis=1)
a_sdm = a_sdm.div(row_sums, axis=0)

print(f"  SDM a_curves shape: {a_sdm.shape}")
print(f"  SDM species: {len(a_sdm)}")

In [ ]:
# Shared pair universe — SDM pollinator coverage only
plants_all = (
    set(Vf_df.index) & set(f_curves_df.index) &
    set(Vd4_df.index) & set(Vd15_df.index) & set(Vfp_df.index)
)
# SDM restricts pollinator pool to those with SDM curves
polls_all = set(Vp_df.index) & set(a_sdm.index)

print(f"Plants in all models:    {len(plants_all)}")
print(f"Pollinators (SDM):       {len(polls_all)}")
# Expected: ~1,275 (SDM 1,615 species, minus those not in Vp)

globi = pd.read_csv(GLOBI_PATH)
globi = globi.rename(columns={
    "sourceTaxonName": "plant_species",
    "targetTaxonName": "pollinator_species"
})
globi = globi.dropna(subset=["plant_species", "pollinator_species"])
globi = globi[["plant_species", "pollinator_species"]].drop_duplicates()

globi_shared = globi[
    globi["plant_species"].isin(plants_all) &
    globi["pollinator_species"].isin(polls_all)
].reset_index(drop=True)

plant_list   = sorted(plants_all)
poll_list    = sorted(polls_all)
positive_set = set(zip(globi_shared["plant_species"], globi_shared["pollinator_species"]))
n_neg        = len(globi_shared) * NEG_RATIO

print(f"Positive pairs (SDM universe): {len(globi_shared)}")
# Expected: ~122 (vs 139 under GBIF)

In [ ]:
# 5-seed evaluation with SDM curves
model_names = [
    "Spatial Baseline-SDM",
    "ANTHEIA-Scalar-SDM",
    "ANTHEIA-4D-SDM",
    "ANTHEIA-15D-SDM",
    "ANTHEIA-PMf-SDM",
]
seed_results = {name: {"roc": [], "pr": []} for name in model_names}

for seed in SEEDS:
    rng_s = np.random.default_rng(seed)

    negatives_s = []
    while len(negatives_s) < n_neg:
        pl_sample = rng_s.choice(plant_list, size=n_neg * 2)
        po_sample = rng_s.choice(poll_list,  size=n_neg * 2)
        for pl, po in zip(pl_sample, po_sample):
            if (pl, po) not in positive_set:
                negatives_s.append((pl, po))
            if len(negatives_s) >= n_neg:
                break

    neg_df_s = pd.DataFrame(negatives_s, columns=["plant_species", "pollinator_species"])
    pairs_s  = pd.concat([
        globi_shared.assign(label=1),
        neg_df_s.assign(label=0)
    ], ignore_index=True)
    y_s = pairs_s["label"].values

    rows = {name: [] for name in model_names}
    for _, row in pairs_s.iterrows():
        pl, po = row["plant_species"], row["pollinator_species"]
        vf    = Vf_df.loc[pl].values
        vp    = Vp_df.loc[po].values
        n     = compute_N(pl, po)
        f     = f_curves_df.loc[pl].values
        a     = a_sdm.loc[po].values       # SDM curves — model-predicted
        delta = np.minimum(f, a).sum()
        vd4   = Vd4_df.loc[pl].values
        vd15  = Vd15_df.loc[pl].values
        vfp   = Vfp_df.loc[pl].values

        rows["Spatial Baseline-SDM"].append(np.concatenate([vf, vp, [n]]))
        rows["ANTHEIA-Scalar-SDM"].append(np.concatenate([vf, vp, [n, delta]]))
        rows["ANTHEIA-4D-SDM"].append(np.concatenate([vf, vp, [n], vd4]))
        rows["ANTHEIA-15D-SDM"].append(np.concatenate([vf, vp, [n], vd15]))
        rows["ANTHEIA-PMf-SDM"].append(np.concatenate([vfp, vp, [n]]))

    X_matrices = {name: np.array(rows[name]) for name in model_names}

    idx = np.arange(len(pairs_s))
    tr_idx, te_idx = train_test_split(
        idx, test_size=0.2, random_state=seed, stratify=y_s
    )

    for name in model_names:
        X_m = X_matrices[name]
        clf = LogisticRegression(max_iter=1000, random_state=seed)
        clf.fit(X_m[tr_idx], y_s[tr_idx])
        y_prob = clf.predict_proba(X_m[te_idx])[:, 1]
        roc = roc_auc_score(y_s[te_idx], y_prob)
        pr  = average_precision_score(y_s[te_idx], y_prob)
        seed_results[name]["roc"].append(roc)
        seed_results[name]["pr"].append(pr)

    print(f"  Seed {seed} done.")

In [ ]:
# Print and save SDM results
print(f"\n{'Model':<26} {'ROC-AUC':>20} {'PR-AUC':>20}")
print("-" * 70)

rows_out = []
for name in model_names:
    roc_mean = np.mean(seed_results[name]["roc"])
    roc_std  = np.std(seed_results[name]["roc"])
    pr_mean  = np.mean(seed_results[name]["pr"])
    pr_std   = np.std(seed_results[name]["pr"])
    print(f"{name:<26} {roc_mean:.4f} ± {roc_std:.4f}   {pr_mean:.4f} ± {pr_std:.4f}")
    rows_out.append({
        "Model"        : name,
        "ROC-AUC Mean" : round(roc_mean, 4),
        "ROC-AUC Std"  : round(roc_std,  4),
        "PR-AUC Mean"  : round(pr_mean,  4),
        "PR-AUC Std"   : round(pr_std,   4),
    })

sdm_results_df = pd.DataFrame(rows_out)
sdm_results_df.to_csv(STAGE6 / "sdm_model_results.csv", index=False)
print(f"\nSaved sdm_model_results.csv")
print()
print("Note: lower absolute performance vs GBIF results reflects smaller")
print("pair universe (122 vs 139 positive pairs) due to SDM coverage.")
print("Model ranking shift (ANTHEIA-PMf-SDM leading) suggests richer")
print("plant-side representations benefit more from cleaner pollinator signal.")
print()
print("TODO: rerun with full 25,466-species SDM when available (September).")